# Forward-modelled $C_\ell$ vs theory: WL, GC, CMB lensing & ISW

This example validates the forward model's line-of-sight **projection kernels** at
the angular power-spectrum level for all four probes it produces:

| probe | map | kernel |
|---|---|---|
| **weak lensing (WL)** | convergence $\kappa$ | lensing efficiency $\times\,n(z)$ |
| **galaxy clustering (GC)** | galaxy overdensity $\delta_g$ | $b_1\,n(z)$ |
| **CMB lensing** | convergence $\kappa$ | lensing efficiency to $z_\star$ |
| **ISW** | $\Delta T/T$ | $3\Omega_m(H_0/c)^3 E(z)(1-f)\chi^2$ |

**The idea.** We generate cross-correlated **lognormal** density shells *from the
cosmology alone* (GLASS, fed CCL matter angular spectra), forward-model the four
probes, measure their auto-$C_\ell$ with `anafast`, and overlay the matching
**CCL** prediction. Because the shells are built from the cosmology's own matter
$C_\ell$, the projected probe spectra should reproduce CCL *by construction* — so
any residual is a property of the projection machinery, not of the input physics.
This is the multi-probe, theory-side companion to
[`05_nbody_vs_lognormal_cls.ipynb`](05_nbody_vs_lognormal_cls.ipynb) (which
compares N-body vs lognormal vs theory for WL alone).

**Choices that make it apples-to-apples**

* **Limber theory.** `ccl.angular_cl(..., l_limber=-1)` (the default) uses the
  Limber approximation at every $\ell$ — the same approximation the shell
  projection embodies.
* **No pixel window.** GLASS synthesises each shell from $a_{\ell m}$, so the maps
  carry no HEALPix pixel window; we neither apply one nor divide the theory by one.
* **Thin low-$z$ shells.** The lensing kernels are discretised per shell and only
  converge to the continuous CCL tracer when the shells are thin (see the WL
  takeaway). We use $\Delta z = 0.04$ below $z=3$ — comparable to a real N-body
  lightcone such as CosmoGrid (~0.05).
* **Monte-Carlo average.** A single full-sky realization is cosmic-variance
  scattered — badly so for the large-scale ISW — so we average several and show
  the error on the mean.

Requires the optional `kassiopeia[lognormal]` extra (GLASS) and `pyccl`.

In [ ]:
import time
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
import pyccl as ccl

import kassiopeia as kp
import glass

# ---- configuration -------------------------------------------------------
NSIDE = 512                 # HEALPix resolution
LMAX  = 1100                # band-limit for everything (generation, projection,
                            # measurement, theory, top bandpower). Deliberately set
                            # to the top bandpower rather than 3*NSIDE-1: with the
                            # shells synthesised band-limited at LMAX there is no
                            # power above l=1100 to alias below 2*NSIDE=1024.
N_MC  = 12                  # full-sky realizations to average (lower me for a quick run)
Z_ISW = 3.0                 # late-time ISW: project on shells below this z only

# Eisenstein-Hu linear transfer (no CAMB needed) + halofit nonlinear P(k); the
# shells and the theory are built from this *same* cosmology, so they are
# self-consistent regardless of the absolute accuracy of the prescription.
cosmo = kp.Cosmology(Omega_m=0.30, Omega_b=0.05, h=0.67, sigma8=0.81, n_s=0.96,
                     transfer_function="eisenstein_hu")
T_CMB = cosmo.ccl["T_CMB"]
print(cosmo, f" T_CMB = {T_CMB:.4f} K")

## 1. Cosmology and shells

We tile redshift with **thin** shells ($\Delta z = 0.04$) out to $z=3$ — fine
enough for the WL and ISW kernels to converge — and a **coarse geometric tail**
out to the last-scattering surface ($z\simeq1100$) for the CMB-lensing integral
(that kernel is broad and smooth, so wide bins there are harmless).

The cross-shell matter spectra and their lognormal Gaussianisation depend only on
the cosmology and the binning — *not* the random seed — so we compute them **once**
and redraw only the field realization inside the Monte-Carlo loop. (The one-line
equivalent for a single realization is `kp.ShellStack.from_lognormal(cosmo,
z_edges, nside=NSIDE)`.)

In [ ]:
# tapered redshift grid
z_thin  = np.arange(0.0, 3.0 + 1e-9, 0.04)        # thin: lensing/ISW accuracy
z_tail  = np.geomspace(3.0, 1100.0, 26)[1:]       # coarse tail: CMB-lensing reach
z_edges = np.concatenate([z_thin, z_tail])
z_lo, z_hi = z_edges[:-1], z_edges[1:]
z_mid = 0.5 * (z_lo + z_hi)
n_shells = len(z_mid)
n_thin = int((z_mid < 3.0).sum())
print(f"{n_shells} shells: {n_thin} thin (dz=0.04, z<3)"
      f" + {n_shells - n_thin} tail (z<1100)")


def matter_spectra(windows, lmax):
    """Pairwise matter angular spectra for the GLASS tophat windows, in the field
    order GLASS expects (i, i-1, ..., 0). Each window is a unit-bias matter-density
    Limber projection -- exactly what drives `ShellStack.from_lognormal`, and the
    same matter C_l the probe theory below is built from."""
    ells = np.arange(lmax + 1.0)
    trs = []
    for w in windows:
        za = np.asarray(w.za, dtype=float)
        trs.append(ccl.NumberCountsTracer(cosmo.ccl, has_rsd=False,
                                          dndz=(za, np.asarray(w.wa, dtype=float)),
                                          bias=(za, np.ones_like(za))))
    return [ccl.angular_cl(cosmo.ccl, trs[i], trs[j], ells)
            for i in range(len(windows)) for j in range(i, -1, -1)]


# one-time GLASS setup (cosmology + binning only)
t0 = time.time()
win_dz  = min(0.01, float(np.min(np.diff(z_edges))) / 8.0)
windows = glass.tophat_windows(z_edges, dz=win_dz)
spectra = matter_spectra(windows, LMAX)
fields  = glass.lognormal_fields(windows)
gls     = glass.regularized_spectra(
    glass.compute_gaussian_spectra(fields, spectra), method="clip")
print(f"one-time GLASS setup: {time.time() - t0:.1f}s")


def generate_shells(rng):
    """One cross-correlated lognormal realization, as a ShellStack."""
    npix = hp.nside2npix(NSIDE)
    delta = np.empty((n_shells, npix))
    for i, fld in enumerate(glass.generate(fields, gls, NSIDE, rng=rng)):
        delta[i] = np.asarray(fld)
    return kp.ShellStack.from_overdensity(delta, z_lo, z_hi, nside=NSIDE)

## 2. Redshift distributions and probes

One representative tomographic bin per two-point probe keeps each panel readable.
We use the DES Y3 $n(z)$ if the data file is present, otherwise simple Gaussian
$n(z)$ so the notebook runs anywhere. The four probes are then projected in a
single pass over the shells.

In [ ]:
DATA = "/path/to/data"  # <-- edit me (optional: without it a synthetic n(z) is used)
try:
    from kassiopeia.loaders import des_y3
    _nz = des_y3.load_nz(f"{DATA}/n_z/y3_redshift_distributions.h5")
    src, lens, nz_label = _nz["source"][3], _nz["lens"][2], "DES Y3"
except Exception:
    z = np.linspace(0.01, 3.0, 400)
    g = lambda m, s: np.exp(-0.5 * ((z - m) / s) ** 2)
    src, lens, nz_label = kp.NumberDensity(z, g(0.9, 0.22)), kp.NumberDensity(z, g(0.5, 0.11)), "synthetic"

b1 = 1.7   # linear galaxy bias of the lens sample
print(f"{nz_label} n(z):  source <z> = {src.mean_z():.2f},  lens <z> = {lens.mean_z():.2f}")

probes = dict(
    weak_lensing      = kp.WeakLensing(nz=[src]),
    galaxy_clustering = kp.GalaxyClustering(nz=[lens], bias=kp.LinearBias(b1=b1)),
    cmb_lensing       = kp.CMBLensing(z_source=1090.0),
    isw               = kp.ISW(),
)
keys = list(probes)

## 3. Theory: CCL tracers

Each forward-model probe maps onto a CCL tracer with the *same* $n(z)$, bias and
source plane. Two conventions to match:

* **CMB lensing / WL** convergence and **GC** overdensity compare directly to
  `CMBLensingTracer`, `WeakLensingTracer` (shear $E$-mode $=\kappa$) and
  `NumberCountsTracer` (`has_rsd=False`, no magnification — the forward model's
  linear GC has neither).
* **ISW** — CCL's `ISWTracer` kernel carries a factor $T_{\rm CMB}$ (it returns
  $\Delta T$ in kelvin), whereas the forward model returns the dimensionless
  $\Delta T/T_{\rm CMB}$. So the CCL ISW spectrum is divided by $T_{\rm CMB}^2$.
  We also cap its integral at `Z_ISW` to match the shells we feed the projector.

In [ ]:
ell = np.arange(LMAX + 1)
l_th = ell[2:]

def theory_cl(tr1, tr2=None):
    out = np.zeros(LMAX + 1)
    out[2:] = ccl.angular_cl(cosmo.ccl, tr1, tr2 or tr1, l_th)   # l_limber=-1 default => Limber
    return out

z_ones = lambda nd: np.ones_like(nd.z)
theory = dict(
    weak_lensing      = theory_cl(ccl.WeakLensingTracer(cosmo.ccl, dndz=(src.z, src.nz))),
    galaxy_clustering = theory_cl(ccl.NumberCountsTracer(
        cosmo.ccl, has_rsd=False, dndz=(lens.z, lens.nz), bias=(lens.z, b1 * z_ones(lens)))),
    cmb_lensing       = theory_cl(ccl.CMBLensingTracer(cosmo.ccl, z_source=1090.0)),
    isw               = theory_cl(ccl.ISWTracer(cosmo.ccl, z_max=Z_ISW)) / T_CMB**2,
)

# 11 log-spaced bandpowers from l=8 to l=1100 (shared l-grid, common to all probes)
bin_edges = np.unique(np.geomspace(8, LMAX, 12).astype(int))
def bandpowers(cl):
    bl, bc = [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        m = (ell >= lo) & (ell < hi)
        if m.any():
            bl.append(ell[m].mean()); bc.append(cl[m].mean())
    return np.array(bl), np.array(bc)
bl = bandpowers(theory["weak_lensing"])[0]
print(f"{len(bl)} bandpowers from l={bl[0]:.0f} to l={bl[-1]:.0f}")

## 4. Monte-Carlo measurement

For each realization: draw new lognormal shells, project WL/GC/CMB-lensing in one
pass, and project the ISW separately on the **low-$z$ shells only**. The high-$z$
tail shells the CMB-lensing integral needs carry an enormous $\chi^2\Delta\chi$
*and* a **radiation-driven** $(1-f)\neq0$; left in, they would swamp the
dark-energy (late-time) ISW with spurious early-time power.

In [ ]:
band_cls = {k: [] for k in keys}
t0 = time.time()
for r in range(N_MC):
    shells = generate_shells(np.random.default_rng(1000 + r))
    fwd = kp.ForwardModel(cosmo, shells, lmax=LMAX)
    m = fwd.project(weak_lensing=probes["weak_lensing"],
                    galaxy_clustering=probes["galaxy_clustering"],
                    cmb_lensing=probes["cmb_lensing"])
    meas = {
        "weak_lensing":      hp.anafast(m.weak_lensing.kappa[0], lmax=LMAX),
        "galaxy_clustering": hp.anafast(m.galaxy_clustering.delta_g[0], lmax=LMAX),
        "cmb_lensing":       hp.anafast(m.cmb_lensing.kappa, lmax=LMAX),
    }
    low = z_mid < Z_ISW
    isw_stack = kp.ShellStack.from_overdensity(
        shells.data[low], z_lo[low], z_hi[low], nside=NSIDE)
    theta = kp.ForwardModel(cosmo, isw_stack, lmax=LMAX).project(
        isw=probes["isw"]).isw.theta
    meas["isw"] = hp.anafast(theta, lmax=LMAX)

    for k in keys:
        band_cls[k].append(bandpowers(meas[k])[1])
    print(f"  realization {r + 1:2d}/{N_MC}   ({time.time() - t0:5.0f}s elapsed)")

band_cls = {k: np.array(v) for k, v in band_cls.items()}          # (N_MC, n_band)
mean_cl  = {k: band_cls[k].mean(0) for k in keys}
err_cl   = {k: band_cls[k].std(0, ddof=1) / np.sqrt(N_MC) for k in keys}  # error on mean
print(f"done: {N_MC} realizations in {time.time() - t0:.0f}s")

## 5. Results

Top row: forward-model bandpowers (points, error on the mean) over the CCL Limber
prediction (line). Bottom row: their ratio, with the $\pm5\%$ band shaded. The
ISW panels stop at $\ell\sim150$, beyond which the signal falls below the
realization noise.

In [ ]:
titles = {
    "weak_lensing":      r"weak lensing  $C_\ell^{\kappa\kappa}$",
    "galaxy_clustering": r"galaxy clustering  $C_\ell^{gg}$",
    "cmb_lensing":       r"CMB lensing  $C_\ell^{\kappa\kappa}$",
    "isw":               r"ISW  $C_\ell^{\Theta\Theta}$  $(\Delta T/T)$",
}
lmax_probe  = {"weak_lensing": LMAX, "galaxy_clustering": LMAX,
               "cmb_lensing": LMAX, "isw": 150}
ratio_ylim  = {"weak_lensing": (0.85, 1.15), "galaxy_clustering": (0.85, 1.15),
               "cmb_lensing": (0.85, 1.15), "isw": (0.5, 1.5)}

fig, axes = plt.subplots(2, 4, figsize=(16, 6.4),
                         gridspec_kw=dict(height_ratios=[2.4, 1]))
for j, k in enumerate(keys):
    sel = bl <= lmax_probe[k]
    bc_th = bandpowers(theory[k])[1]
    top, bot = axes[0, j], axes[1, j]

    top.loglog(bl[sel], bc_th[sel], "k-", lw=1.8, label="CCL theory (Limber)")
    top.errorbar(bl[sel], mean_cl[k][sel], yerr=err_cl[k][sel], fmt="o", ms=4,
                 color="C3", capsize=2, label=f"forward model (mean of {N_MC})")
    top.set_title(titles[k], fontsize=10)
    top.legend(fontsize=7.5, loc="lower left")
    if j == 0:
        top.set_ylabel(r"$C_\ell$")

    ratio = mean_cl[k][sel] / bc_th[sel]
    bot.axhspan(0.95, 1.05, color="0.88", zorder=0)
    bot.axhline(1.0, color="k", lw=0.8)
    bot.errorbar(bl[sel], ratio, yerr=err_cl[k][sel] / bc_th[sel], fmt="o", ms=4,
                 color="C3", capsize=2)
    bot.set_xscale("log")
    bot.set_ylim(*ratio_ylim[k])
    bot.set_xlabel(r"$\ell$")
    if j == 0:
        bot.set_ylabel("measured / theory")

    # x-limits matched to the measured bandpower range for this probe
    xlo, xhi = bin_edges[0] * 0.9, bin_edges[int(sel.sum())] * 1.1
    top.set_xlim(xlo, xhi)
    bot.set_xlim(xlo, xhi)

fig.suptitle(rf"Forward-modelled $C_\ell$ vs CCL theory   "
             rf"(NSIDE={NSIDE}, {n_shells} shells, {nz_label} $n(z)$, "
             rf"mean of {N_MC} realizations)", y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# numeric summary: inverse-variance mean ratio over a clean band per probe
band = {"weak_lensing": (60, 400), "galaxy_clustering": (60, 400),
        "cmb_lensing": (60, 400), "isw": (4, 40)}
print("             probe     l band   mean(meas/theory)")
for k in keys:
    lo, hi = band[k]
    sel = (bl >= lo) & (bl <= hi)
    bc_th = bandpowers(theory[k])[1]
    rat = mean_cl[k][sel] / bc_th[sel]
    rerr = err_cl[k][sel] / bc_th[sel]
    w = 1.0 / rerr**2
    mu = np.sum(w * rat) / np.sum(w)
    sd = np.sqrt(1.0 / np.sum(w))
    print(f"{k:>18}  {lo:4d}-{hi:<4d}   {mu:6.3f} +/- {sd:.3f}")

## Takeaways

* **WL, GC and CMB lensing** track the CCL Limber prediction to a few percent
  across the whole band; the bandpower scatter is consistent with the plotted
  error on the mean (cosmic variance $/\sqrt{N_{\rm MC}}$). With the shells built
  from the cosmology's own matter $C_\ell$, this is the projection kernels
  reproducing theory by construction — the intended validation.

* **Weak lensing needs thin shells.** The line-of-sight lensing weight is
  evaluated per shell, and the discretised kernel only converges to the continuous
  CCL `WeakLensingTracer` as the shells get thin — the effect is largest for
  *low-$z$ sources*, where the efficiency $\chi(\chi_s-\chi)/\chi_s$ varies fastest
  across a bin. Coarse shells ($\Delta z\gtrsim0.15$) overpredict WL by tens of
  percent; $\Delta z\simeq0.04$ (this notebook) and a real N-body lightcone's
  ~0.05 shells (as in [`05`](05_nbody_vs_lognormal_cls.ipynb), where WL matches
  cleanly) are both fine. CMB lensing is immune because its source plane sits far
  behind every shell.

* **ISW is the delicate one.**
  - *Units:* the forward model returns $\Delta T/T_{\rm CMB}$, CCL's `ISWTracer`
    returns $\Delta T$ in kelvin, hence the $1/T_{\rm CMB}^2$ on the theory.
  - *Redshift range:* the late-time (dark-energy) ISW lives at $z\lesssim2$. The
    high-$z$ tail shells must be excluded — there $(1-f)$ is reawakened by
    *radiation*, not dark energy, and $\chi^2\Delta\chi$ is huge, so those shells
    would inject large spurious power. This is early-time ISW, which this
    Limber+lognormal treatment does not model.
  - *Statistics:* the signal sits at low $\ell$, where there are few modes, so the
    auto-spectrum is cosmic-variance limited even full-sky — hence the
    Monte-Carlo average and the wider error bars. A residual few-percent offset at
    the lowest $\ell$ also reflects the $1/[\ell(\ell+1)]$ vs $1/(\ell+\tfrac12)^2$
    inverse-Laplacian convention and the Limber approximation weakening at
    $\ell\sim2$. In practice the ISW is measured via its **cross-correlation**
    with a low-$z$ tracer (galaxies or CMB lensing); the same maps support that
    higher-SNR test directly.

* **Method.** Limber theory (`l_limber=-1`) matches the shell projection; the
  GLASS maps carry no pixel window, so none is applied to either side; everything
  is full-sky (no mask), so the only scatter is cosmic variance.